In [6]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# 1. Carga de datos con ruta relativa segura
ARCHIVO_DATASET = "banco_transacciones.csv"  # o "banco_transacciones_3.csv"
df = pd.read_csv(ARCHIVO_DATASET)
df.columns = df.columns.str.strip()

# 2. Detección dinámica de columnas
col_categoria = (
    "Categoria_Transaccion"
    if "Categoria_Transaccion" in df.columns
    else "categoria_nombre"
)
col_descripcion = (
    "Descripcion_Transaccion"
    if "Descripcion_Transaccion" in df.columns
    else "concepto"
)

# 3. Diccionario de Clasificación
clasificacion = {
    "Centro comercial": "Alimentación",
    "Comida rápida": "Alimentación",
    "Supermercado": "Alimentación",
    "Restaurantes": "Alimentación",
    "Taxi": "Transporte",
    "Bus": "Transporte",
    "Metro": "Transporte",
    "Gasolina": "Transporte",
    "Mantenimiento": "Transporte",
    "Multas": "Transporte",
    "Seguros": "Salud",
    "Farmacia": "Salud",
    "Renta": "Vivienda",
    "Colegiaturas": "Educación",
    "Colegiatura": "Educación",
    "Viajes": "Ocio",
    "Streaming": "Ocio",
    "Videojuegos": "Ocio",
    "Salidas": "Ocio",
    "Ropa": "Ocio",
    "Agua": "Servicios",
    "Electricidad": "Servicios",
    "Gas": "Servicios",
    "Internet y Telefonía Hogar": "Servicios",
    "Telefonía Móvil": "Servicios",
    "Tarjetas de crédito": "Deudas",
    "Préstamos bancarios": "Deudas",
    "Ahorro": "Ahorro",
    "Nómina/Ingresos": "Ingreso",
}

# 4. Mapeo
df["Macro_categoria"] = df[col_categoria].map(clasificacion).fillna("Otros")
df[col_descripcion] = df[col_descripcion].fillna("Gasto Vario").astype(str)

# 5. División Train/Test
X = df[col_descripcion]
y = df["Macro_categoria"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Pipeline de Machine Learning
ml_pipeline = Pipeline(
    [
        (
            "convertidor_texto",
            TfidfVectorizer(ngram_range=(1, 2), strip_accents="unicode"),
        ),
        (
            "algoritmo_ia",
            LogisticRegression(random_state=42, max_iter=1000, C=1.0),
        ),
    ]
)

# 7. Entrenamiento y Evaluación
ml_pipeline.fit(X_train, y_train)
y_pred = ml_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

# 8. Guardar artefacto
joblib.dump(ml_pipeline, "modelo_clasificador_salud_financiera.pkl")
print("¡Modelo exportado como 'modelo_clasificador_salud_financiera.pkl'!")

              precision    recall  f1-score   support

Alimentación       1.00      1.00      1.00        27
   Educación       1.00      1.00      1.00        26
        Ocio       1.00      1.00      1.00       128
       Otros       1.00      1.00      1.00       512
       Salud       1.00      1.00      1.00        51
   Servicios       1.00      1.00      1.00        74
  Transporte       1.00      1.00      1.00       157
    Vivienda       1.00      1.00      1.00        25

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

¡Modelo exportado como 'modelo_clasificador_salud_financiera.pkl'!
